#### ***Position Embedding***
##### ***Position Embedding tells the LLM  where the each token in a sequence.***

In [1]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [2]:
with open("Data/positional_embeddings_data.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

In [3]:
print(len(raw_text))

4507


In [4]:
### create a dataset
import torch
from torch.utils.data import DataLoader, Dataset

class GptDataset(Dataset):

    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

        for i in range(0,len(token_ids)-max_length,stride):
            input_batch = token_ids[i:i+max_length]
            target_batch = token_ids[i+1:i+max_length+1]

            self.input_ids.append(torch.tensor(input_batch))
            self.target_ids.append(torch.tensor(target_batch))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]



In [5]:
def create_dataloader(txt,batch_size=4,max_length = 4,stride = 2,shuffle=True,drop_last=True,num_workers = 0):


    dataset = GptDataset(txt,tokenizer,max_length,stride)

    dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)

    return dataloader

In [6]:
dataloader = create_dataloader(raw_text,batch_size = 16,max_length=8,stride = 6,shuffle=False)

batch_iter = iter(dataloader)
first_batch = next(batch_iter)

input,target = first_batch

print("Input:",input)

Input: tensor([[32065,  4981,  2193,  7572,   422,  1588,  6867,   286],
        [ 6867,   286,  2420,    13,   317,  3303,  2746,  9743],
        [ 2746,  9743, 16326,   290, 22974,   703,   883, 16326],
        [  883, 16326,   389,  3519,   284,  1123,   584,    13],
        [  584,    13, 29130, 11525,    67,   654,  2148, 29052],
        [ 2148, 29052, 24612,   286, 16326,    11,   981, 45203],
        [  981, 45203, 11525,    67,   654,  2148,  1321,   546],
        [ 1321,   546,   810,   883, 16326,  1656,   287,   257],
        [  287,   257,  8379,    13,   198,   198, 43467,   262],
        [43467,   262,  2292,   286,   257, 11241,   318,  1593],
        [  318,  1593,   780,   262,  1502,   286,  2456,   460],
        [ 2456,   460,  1487,   262,  3616,   286,   257,  6827],
        [  257,  6827,    13,   383,  6827,   366,  1169,  3290],
        [ 1169,  3290, 26172,   262,  3797,     1,   318,  1180],
        [  318,  1180,   422,   366,  1169,  3797, 26172,   262],
   

In [15]:
vocab_size = 50257

In [16]:
### create a embedding layer
import torch.nn as nn 
out_dim = 512
embedding_layer = nn.Embedding(vocab_size,out_dim)
print(embedding_layer)

Embedding(50257, 512)


In [18]:
## token embedding
token_embedding = embedding_layer(input)
token_embedding.shape

torch.Size([16, 8, 512])

In [12]:
## lets create a position embedding layer using max_length = 8 length of the each row with 512 dimesional.

pos_embedding_layer = nn.Embedding(8,out_dim)
pos_embedding_layer

Embedding(8, 512)

In [22]:
### 0,1,2,3,4,5,6,7
pos_embedding = pos_embedding_layer(torch.arange(8))
pos_embedding.shape

torch.Size([8, 512])

In [21]:
###input Embedding
input_embedding = token_embedding+pos_embedding
input_embedding.shape

torch.Size([16, 8, 512])